# Word2Vec: Efficient Estimation of Word Representations in Vector Space (Mikolov et al., ICLR 2013)
### Triển Khai Từ Con Số 0 (From Scratch) Bằng Python & NumPy

> **Tác giả:** Tomas Mikolov, Kai Chen, Greg Corrado, Jeffrey Dean  
> **Hội nghị:** ICLR 2013 (Scottsdale, Arizona)  
> **Ý nghĩa lịch sử:** Chuyển dịch toàn diện từ biểu diễn One-Hot rời rạc sang không gian vector thực liên tục dày đặc (Dense Embeddings), mở ra kỷ nguyên học biểu diễn ngữ nghĩa cho NLP.

---

## 🎯 Mục Tiêu Bài Thực Hành
1. **Vocabulary & Negative Sampling Table:** Tự xây dựng bộ mã hóa từ vựng và bảng phân phối lấy mẫu âm lũy thừa $3/4$: $P_n(w) \propto U(w)^{0.75}$.
2. **Skip-gram Model:** Lập trình lan truyền xuôi và ngược giải tích của bài toán hồi quy Logistic nhị phân với $K$ mẫu âm tính.
3. **Đại số ngữ nghĩa (Word Analogy):** Thực thi phép suy luận vector $\vec{v}_{\text{King}} - \vec{v}_{\text{Man}} + \vec{v}_{\text{Woman}} \approx \vec{v}_{\text{Queen}}$.
4. **Academic Value Benchmark:** Đối chiếu định lượng sự khác biệt bản chất giữa Kỷ nguyên Tiền-Word2Vec (One-Hot) và Word2Vec.

In [ ]:
import time
import numpy as np
print(f"[✓] Khởi tạo thư viện NumPy: {np.__version__}")

## PHẦN 1: QUẢN LÝ TỪ VỰNG & PHÂN PHỐI LẤY MẪU TIÊU CỰC

Theo Mikolov et al. (2013), phân phối lấy mẫu âm tính tối ưu là phân phối Unigram nâng lũy thừa $3/4$:
$$P_n(w) = \frac{U(w)^{3/4}}{\sum_{w'} U(w')^{3/4}}$$

Phép nâng lũy thừa $0.75$ giúp tăng nhẹ cơ hội xuất hiện của các từ hiếm mà không làm mất tính tự nhiên của dữ liệu.

In [ ]:
class Vocabulary:
    """
    Quản lý từ vựng và bảng lấy mẫu âm tính unigram table.
    """
    def __init__(self, corpus_text, min_count=1):
        self.min_count = min_count
        self.w2i = {}
        self.i2w = {}
        self.word_counts = {}
        self.total_words = 0
        self.vocab_size = 0
        self.sampling_table = []

        self._build_vocab(corpus_text)
        self._build_negative_sampling_table()

    def _build_vocab(self, corpus_text):
        words = corpus_text.lower().replace(".", " ").replace(",", " ").split()
        for w in words:
            self.word_counts[w] = self.word_counts.get(w, 0) + 1
            self.total_words += 1

        idx = 0
        for w, c in self.word_counts.items():
            if c >= self.min_count:
                self.w2i[w] = idx
                self.i2w[idx] = w
                idx += 1

        self.vocab_size = len(self.w2i)

    def _build_negative_sampling_table(self, table_size=100000):
        powered_counts = np.array([self.word_counts[self.i2w[i]] ** 0.75 for i in range(self.vocab_size)])
        probs = powered_counts / np.sum(powered_counts)

        cumulative = np.cumsum(probs)
        table = np.zeros(table_size, dtype=np.int32)
        idx = 0
        for i in range(table_size):
            r = (i + 1) / table_size
            while idx < self.vocab_size - 1 and r > cumulative[idx]:
                idx += 1
            table[i] = idx
        self.sampling_table = table

    def get_negative_samples(self, target_idx, k=5):
        neg_samples = []
        while len(neg_samples) < k:
            rand_idx = self.sampling_table[np.random.randint(len(self.sampling_table))]
            if rand_idx != target_idx:
                neg_samples.append(rand_idx)
        return neg_samples


def generate_skipgram_training_pairs(corpus_text, vocab, window_size=2):
    words = corpus_text.lower().replace(".", " ").replace(",", " ").split()
    word_indices = [vocab.w2i[w] for w in words if w in vocab.w2i]
    pairs = []
    for center_pos, center_idx in enumerate(word_indices):
        start = max(0, center_pos - window_size)
        end = min(len(word_indices), center_pos + window_size + 1)
        for context_pos in range(start, end):
            if context_pos != center_pos:
                pairs.append((center_idx, word_indices[context_pos]))
    return pairs

print("[✓] Đã khởi tạo lớp Vocabulary và hàm generate_skipgram_training_pairs.")

## PHẦN 2: MÔ HÌNH WORD2VEC SKIP-GRAM TỪ CON SỐ 0

Thay vì tính Softmax $V$ chiều tốn kém, hàm mất mát Negative Sampling là:
$$\mathcal{L} = -\log \sigma({v'_{w_O}}^\top v_{w_I}) - \sum_{k=1}^K \log \sigma(-{v'_{n_k}}^\top v_{w_I})$$

- Mẫu dương: Đẩy tích vô hướng lên cao (hướng vector trùng nhau).
- Mẫu âm: Kéo tích vô hướng xuống thấp (hướng vector tách rời nhau).

In [ ]:
class Word2VecScratch:
    def __init__(self, vocab_size, embed_dim=16, learning_rate=0.025):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.lr = learning_rate

        bound = 0.5 / embed_dim
        self.W_in = np.random.uniform(-bound, bound, (vocab_size, embed_dim)).astype(np.float32)
        self.W_out = np.zeros((vocab_size, embed_dim), dtype=np.float32)

    @staticmethod
    def sigmoid(z):
        z_clipped = np.clip(z, -15.0, 15.0)
        return 1.0 / (1.0 + np.exp(-z_clipped))

    def train_pair(self, center_idx, context_idx, neg_indices):
        v_center = self.W_in[center_idx]

        # 1. Mẫu dương
        v_pos = self.W_out[context_idx]
        dot_pos = np.dot(v_pos, v_center)
        sig_pos = self.sigmoid(dot_pos)
        grad_pos = sig_pos - 1.0
        loss = -np.log(sig_pos + 1e-12)

        # 2. Mẫu âm
        v_negs = self.W_out[neg_indices]
        dot_negs = np.dot(v_negs, v_center)
        sig_negs = self.sigmoid(dot_negs)
        grad_negs = sig_negs
        loss += -np.sum(np.log(1.0 - sig_negs + 1e-12))

        # 3. Cập nhật W_in & W_out
        grad_center = grad_pos * v_pos + np.sum(grad_negs[:, None] * v_negs, axis=0)
        self.W_out[context_idx] -= self.lr * (grad_pos * v_center)
        for i, neg_idx in enumerate(neg_indices):
            self.W_out[neg_idx] -= self.lr * (grad_negs[i] * v_center)
        self.W_in[center_idx] -= self.lr * grad_center

        return loss

    def get_word_vector(self, word_idx):
        return self.W_in[word_idx]

print("[✓] Đã khởi tạo Word2VecScratch thành công.")

## PHẦN 3: ĐỘ TƯƠNG ĐỒNG COSINE & BỘ GIẢI ĐẠI SỐ NGỮ NGHĨA

Độ tương đồng Cosine đo góc giữa hai vector trong không gian nhiều chiều:
$$\text{Cosine}(u, v) = \frac{u \cdot v}{\|u\| \|v\|}$$

Phép suy luận quan hệ: $\vec{d} = \vec{v}_{\text{King}} - \vec{v}_{\text{Man}} + \vec{v}_{\text{Woman}}$.

In [ ]:
def cosine_similarity(v1, v2):
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(v1, v2) / (norm1 * norm2)

def solve_analogy(w1, w2, w3, model, vocab, top_k=3):
    for w in [w1, w2, w3]:
        if w not in vocab.w2i:
            return [(f"Từ '{w}' không có trong từ điển", 0.0)]

    idx1, idx2, idx3 = vocab.w2i[w1], vocab.w2i[w2], vocab.w2i[w3]
    target_vec = model.get_word_vector(idx2) - model.get_word_vector(idx1) + model.get_word_vector(idx3)

    exclude = {idx1, idx2, idx3}
    scores = []
    for i in range(vocab.vocab_size):
        if i not in exclude:
            sim = cosine_similarity(target_vec, model.get_word_vector(i))
            scores.append((vocab.i2w[i], sim))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

print("[✓] Đã sẵn sàng cosine_similarity và solve_analogy.")

## PHẦN 4: ĐỐI CHIẾU GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

So sánh phương pháp biểu diễn từ Tiền-Word2Vec (One-Hot Encoding) với Word2Vec (Dense Continuous Embeddings).

In [ ]:
from word2vec_scratch import run_academic_value_comparison

# Chạy kiểm chứng đối chuẩn giá trị học thuật
run_academic_value_comparison()

## 🎓 TỔNG KẾT & DI SẢN CỦA BÀI BÁO WORD2VEC (ICLR 2013)

1. **Chuyển dịch sang Biểu Diễn Dày Đặc (Dense Embeddings):** Loại bỏ hoàn toàn vector One-Hot thưa thớt, biến ngôn ngữ tự nhiên thành không gian hình học đại số có thể tính toán được.
2. **Negative Sampling:** Đơn giản hóa hàm Softmax phức tạp thành phân loại nhị phân, cho phép học trên hàng tỷ từ văn bản trong vài giờ.
3. **Tính Chất Đại Số Tuyệt Đẹp:** Khám phá rằng các hướng vector phản ánh chính xác các khái niệm trừu tượng (Giới tính, Thủ đô - Quốc gia, Thì quá khứ).
4. **Bệ phóng cho NLP hiện đại:** Đặt nền móng vững chắc cho sự ra đời của GloVe, FastText, ELMo, Transformer và LLMs ngày nay.